# 01_Setup_and_Sanity.ipynb
# Abstractive Summarization (BART + LoRA)


In [2]:
# Make sure we're using a GPU:

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [3]:
# Minimal install set for setup + data sanity
!pip -q install datasets transformers evaluate accelerate peft sentencepiece rouge_score huggingface_hub

# Restart runtime if Colab asks.

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00


In [8]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not found in Colab Secrets. Add it in the Secrets panel as HF_TOKEN.")

login(token=hf_token)
print("Logged in to Hugging Face successfully.")

Logged in to Hugging Face successfully.


In [9]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seeds set:", SEED)

Seeds set: 42


In [11]:
from datasets import load_dataset

DATASET_ID = "abisee/cnn_dailymail"
DATASET_CONFIG = "3.0.0"

ds = load_dataset(DATASET_ID, DATASET_CONFIG)

print(ds)
print("Splits:", list(ds.keys()))
print("Columns:", ds["train"].column_names)

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})
Splits: ['train', 'validation', 'test']
Columns: ['article', 'highlights', 'id']


In [12]:
# CNN/DailyMail commonly has: 'article', 'highlights', 'id'
required_cols = {"article", "highlights"}
train_cols = set(ds["train"].column_names)

missing = required_cols - train_cols
if missing:
    raise ValueError(f"Dataset missing expected columns: {missing}")

print("✅ Required columns found:", required_cols)

# Quick look at one sample
sample = ds["train"][0]
print("\n--- SAMPLE KEYS ---")
print(sample.keys())
print("\n--- ARTICLE (first 600 chars) ---")
print(sample["article"][:600])
print("\n--- HIGHLIGHTS (summary) ---")
print(sample["highlights"])

✅ Required columns found: {'article', 'highlights'}

--- SAMPLE KEYS ---
dict_keys(['article', 'highlights', 'id'])

--- ARTICLE (first 600 chars) ---
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," 

--- HIGHLIGHTS (summary) ---
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [13]:
print("Train size:", len(ds["train"]))
print("Validation size:", len(ds["validation"]))
print("Test size:", len(ds["test"]))

Train size: 287113
Validation size: 13368
Test size: 11490


In [14]:
import pandas as pd

def length_stats(split_name, n=5000):
    # sample n rows for speed
    data = ds[split_name].select(range(min(n, len(ds[split_name]))))
    df = pd.DataFrame({
        "article_len_chars": [len(x) if x is not None else 0 for x in data["article"]],
        "summary_len_chars": [len(x) if x is not None else 0 for x in data["highlights"]],
    })
    return df.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

for split in ["train", "validation", "test"]:
    print(f"\n=== {split.upper()} (sampled) LENGTH STATS ===")
    display(length_stats(split, n=5000))


=== TRAIN (sampled) LENGTH STATS ===


,article_len_chars,summary_len_chars
count,5000.000000,5000.0000
mean,3663.637600,259.6762
std,1763.959298,44.2330
min,106.000000,65.0000
50%,3416.000000,267.0000
75%,4837.000000,298.0000
90%,6055.000000,313.0000
95%,6911.000000,321.0000
99%,8441.430000,333.0000
max,11265.000000,455.0000



=== VALIDATION (sampled) LENGTH STATS ===


,article_len_chars,summary_len_chars
count,5000.000000,5000.000000
mean,3266.267000,272.126000
std,1897.874477,107.564108
min,245.000000,56.000000
50%,2756.000000,262.000000
75%,4284.250000,320.000000
90%,5989.300000,392.000000
95%,7190.100000,443.000000
99%,9184.680000,591.010000
max,11010.000000,3264.000000



=== TEST (sampled) LENGTH STATS ===


,article_len_chars,summary_len_chars
count,5000.000000,5000.000000
mean,3361.388800,274.837800
std,1947.815256,112.175906
min,293.000000,51.000000
50%,2836.000000,264.000000
75%,4397.500000,320.000000
90%,6156.000000,391.000000
95%,7438.400000,449.050000
99%,9354.050000,584.010000
max,11655.000000,2882.000000


In [15]:
from transformers import AutoTokenizer

BASE_MODEL_ID = "facebook/bart-large-cnn"  # your baseline + base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

def token_lengths(split_name, n=2000):
    data = ds[split_name].select(range(min(n, len(ds[split_name]))))
    art_lens = []
    sum_lens = []
    for a, s in zip(data["article"], data["highlights"]):
        if a is None or s is None:
            continue
        art_lens.append(len(tokenizer(a, truncation=False)["input_ids"]))
        sum_lens.append(len(tokenizer(s, truncation=False)["input_ids"]))
    return pd.Series(art_lens), pd.Series(sum_lens)

art_lens, sum_lens = token_lengths("train", n=2000)

print("Article token length (train sample) quantiles:")
display(art_lens.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nSummary token length (train sample) quantiles:")
display(sum_lens.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Article token length (train sample) quantiles:


,0
count,2000.000000
mean,759.146000
std,368.662787
min,29.000000
50%,708.000000
75%,997.000000
90%,1261.100000
95%,1430.100000
99%,1821.020000
max,2268.000000



Summary token length (train sample) quantiles:


,0
count,2000.000000
mean,58.178500
std,10.411428
min,14.000000
50%,59.000000
75%,66.000000
90%,71.000000
95%,75.000000
99%,81.010000
max,101.000000


In [17]:
# As 90-95% of articles are <= 1024 tokens, I choose 1024.

MAX_INPUT_TOKENS = 1024
MAX_TARGET_TOKENS = 128

print("Chosen truncation:")
print("MAX_INPUT_TOKENS =", MAX_INPUT_TOKENS)
print("MAX_TARGET_TOKENS =", MAX_TARGET_TOKENS)

Chosen truncation:
MAX_INPUT_TOKENS = 1024
MAX_TARGET_TOKENS = 128


In [19]:
# - Token EDA gives us "typical" lengths, but we must confirm truncation works as intended.
# - We check a few examples to see:
#   1) How long the article/summary is *without* truncation
#   2) What length it becomes *with* truncation
# - This prevents silent bugs where your max_length isn't being applied properly.

def show_truncation_effect(indices=(0, 25, 100)):
    for idx in indices:
        a = ds["train"][idx]["article"]
        s = ds["train"][idx]["highlights"]

        # Full tokenization (no truncation) to measure the real length
        a_full_ids = tokenizer(a, truncation=False)["input_ids"]
        s_full_ids = tokenizer(s, truncation=False)["input_ids"]

        # Truncated tokenization (what training will actually use)
        a_trunc_ids = tokenizer(a, truncation=True, max_length=MAX_INPUT_TOKENS)["input_ids"]
        s_trunc_ids = tokenizer(s, truncation=True, max_length=MAX_TARGET_TOKENS)["input_ids"]

        print(f"\n--- Sample idx={idx} ---")
        print(f"Article tokens: full={len(a_full_ids):4d} -> trunc={len(a_trunc_ids):4d} (cap={MAX_INPUT_TOKENS})")
        print(f"Summary tokens: full={len(s_full_ids):4d} -> trunc={len(s_trunc_ids):4d} (cap={MAX_TARGET_TOKENS})")

show_truncation_effect(indices=(0, 25, 100))


--- Sample idx=0 ---
Article tokens: full= 565 -> trunc= 565 (cap=1024)
Summary tokens: full=  52 -> trunc=  52 (cap=128)

--- Sample idx=25 ---
Article tokens: full= 809 -> trunc= 809 (cap=1024)
Summary tokens: full=  46 -> trunc=  46 (cap=128)

--- Sample idx=100 ---
Article tokens: full= 783 -> trunc= 783 (cap=1024)
Summary tokens: full=  57 -> trunc=  57 (cap=128)


In [20]:
# - CNN/DailyMail is usually clean, so we do NOT want aggressive filtering.
# - But it’s still smart to check for obvious junk:
#   - missing article/summary
#   - extremely short article/summary (often bad training signal)
#
# Outcome:
# - If the % is tiny, we can skip filtering entirely for now.
# - If it’s non-trivial, we can add a simple filter later in training notebook.

MIN_ARTICLE_CHARS = 200
MIN_SUMMARY_CHARS = 30
CHECK_N = 5000  # sample size for quick scan (fast)

def scan_bad_examples(split_name="train", n=5000):
    data = ds[split_name].select(range(min(n, len(ds[split_name]))))
    bad = 0
    reasons = {"missing": 0, "too_short": 0}

    for a, s in zip(data["article"], data["highlights"]):
        if a is None or s is None:
            bad += 1
            reasons["missing"] += 1
            continue
        if len(a) < MIN_ARTICLE_CHARS or len(s) < MIN_SUMMARY_CHARS:
            bad += 1
            reasons["too_short"] += 1

    total = len(data)
    return bad, total, reasons

bad, total, reasons = scan_bad_examples("train", n=CHECK_N)
print(f"Bad examples in train sample: {bad}/{total} ({bad/total:.2%})")
print("Reason breakdown:", reasons)

# If this is <0.5% (often is),  we can skip filtering for now.

Bad examples in train sample: 2/5000 (0.04%)
Reason breakdown: {'missing': 0, 'too_short': 2}


In [23]:
# Why this exists:
# - You want Notebook 2/3/4/5 (baseline/LoRA/eval/demo) to reuse the SAME decisions:
#   dataset id/config, base model id, truncation lengths, fields mapping, seed, etc.
# - So we save one JSON file: project_config.json
# - Every future notebook will load it and stay consistent.

import json
from datetime import datetime, UTC

# Build config dictionary
project_config = {
    "project_name": "abstractive_summarizer_bart_lora",
    "created_utc": datetime.now(UTC).isoformat(),
    "seed": SEED,

    "dataset": {
        "id": DATASET_ID,
        "config": DATASET_CONFIG,
        "splits": ["train", "validation", "test"],
        "text_field": "article",
        "summary_field": "highlights"
    },

    "model": {
        "base_model_id": BASE_MODEL_ID,
        "tokenizer_id": BASE_MODEL_ID
    },

    "tokenization": {
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_target_tokens": MAX_TARGET_TOKENS,
        "padding": "max_length",
        "truncation": True
    },

    "data_sanity": {
        "min_article_chars": MIN_ARTICLE_CHARS,
        "min_summary_chars": MIN_SUMMARY_CHARS,
        "scan_sample_n": CHECK_N
    },

    "notes": [
        "Mapping: article -> highlights",
        "Truncation chosen from token-length EDA sample"
    ]
}

CONFIG_LOCAL_PATH = "project_config.json"

# Save locally (always overwrite)
with open(CONFIG_LOCAL_PATH, "w") as f:
    json.dump(project_config, f, indent=2)

print("✅ Local config updated:", CONFIG_LOCAL_PATH)


# =========================
# Mount Google Drive if not mounted
# =========================
from google.colab import drive
import os

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

DRIVE_DIR = "/content/drive/MyDrive/abstractive_summarizer_bart_lora"
os.makedirs(DRIVE_DIR, exist_ok=True)

CONFIG_DRIVE_PATH = f"{DRIVE_DIR}/project_config.json"

# Overwriting safely
import shutil
shutil.copy(CONFIG_LOCAL_PATH, CONFIG_DRIVE_PATH)

print("✅ Drive config updated:", CONFIG_DRIVE_PATH)


import torch

meta = {
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
}

META_LOCAL = "run_meta.json"
META_DRIVE = f"{DRIVE_DIR}/run_meta.json"

with open(META_LOCAL, "w") as f:
    json.dump(meta, f, indent=2)

shutil.copy(META_LOCAL, META_DRIVE)

print("✅ Runtime metadata updated")

✅ Local config updated: project_config.json
Drive already mounted.
✅ Drive config updated: /content/drive/MyDrive/abstractive_summarizer_bart_lora/project_config.json
✅ Runtime metadata updated


## Notebook: Setup and Dataset Sanity

In this notebook I prepared the environment and verified the dataset before starting model experiments.

- Set up the Colab environment, installed required libraries (`transformers`, `datasets`, `peft`, etc.), and verified GPU availability.
- Logged into Hugging Face using my access token to enable downloading datasets and pretrained models.
- Loaded the **CNN/DailyMail dataset (config 3.0.0)** and confirmed the expected mapping: `article` → `highlights`.
- Performed lightweight EDA to inspect article and summary lengths using both characters and tokenizer-based token counts.
- Selected truncation limits of **1024 tokens for input articles** and **128 tokens for summaries**, based on the token length distribution.
- Created and saved a **shared project configuration file (`project_config.json`)** so all later notebooks (baseline, LoRA training, evaluation, demo) use the same settings.